# Train/test EC classification

Load the stored contact graphs, create a reproducible grouped train/test split, label amino-acid residues and contact edges, then fit a simplex-pair graph classifier. The target is the EC prefix chosen in notebook 01. UniProt accession is used as the split group to keep a protein out of both train and test.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
import sys
import warnings

import joblib
import networkx as nx
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import Normalizer
from sklearn.linear_model import SGDClassifier

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

GRAPHICALIZER_SRC = PROJECT_ROOT.parent / "abstractgraph-ecosystem/repos/abstractgraph-graphicalizer/src"
if GRAPHICALIZER_SRC.exists():
    sys.path.insert(0, str(GRAPHICALIZER_SRC))

from abstractgraph_graphicalizer.protein import ProteinLabelGraphicalizer, read_graph_pickle
from simplex_pair_vectorizer import SimplexPairGraphVectorizer

DATA_DIR = PROJECT_ROOT / "data"
EC_LEVEL = 1  # Keep this aligned with notebooks 01 and 02.
MANIFEST_PATH = DATA_DIR / "interim" / f"ec_protein_manifest_level_{EC_LEVEL}.tsv"
STATUS_PATH = DATA_DIR / "processed" / f"graph_build_status_level_{EC_LEVEL}.tsv"
RESULTS_DIR = PROJECT_ROOT / "results"

TEST_SIZE = 0.20
RANDOM_STATE = 42
NODE_LABEL_ALPHABET = "aa20"  # aa20, hp2, charge4, physchem5, chem7, or dayhoff6
BASE_ESTIMATOR_KIND = "linear_sgd"  # linear_sgd or random_forest
MAX_SIMPLEX_DIMENSION = 3
MIN_PAIR_DISTANCE = 0
MAX_PAIR_DISTANCE = 4
NBITS = 14

In [ ]:
if not MANIFEST_PATH.exists() or not STATUS_PATH.exists():
    raise FileNotFoundError("Run notebooks 01 and 02 first to create a manifest and graph status file")

manifest = pd.read_csv(MANIFEST_PATH, sep="\t", dtype=str).fillna("")
status = pd.read_csv(STATUS_PATH, sep="\t", dtype=str).fillna("")
dataset = manifest.merge(status, on="sample_id", how="inner", suffixes=("", "_build"))
dataset = dataset[dataset["status"].isin(["created", "skipped_existing"])].copy()
dataset = dataset[dataset["graph_path"].map(lambda path: Path(path).exists())].reset_index(drop=True)
if dataset.empty:
    raise RuntimeError("No completed graph files found")

graphs = [read_graph_pickle(path) for path in dataset["graph_path"]]
y = dataset["ec_number"].astype(str).to_numpy()
groups = dataset["uniprot_accession"].astype(str).to_numpy()
print(f"Loaded {len(graphs):,} graphs across {len(set(y))} EC labels")
display(dataset.groupby("ec_number").size().rename("samples").to_frame())

In [ ]:
class_counts = pd.Series(y).value_counts().sort_index()
if len(class_counts) < 2:
    raise ValueError("Classification needs at least two EC classes")
if class_counts.min() < 2:
    raise ValueError(
        "At least one selected EC class has fewer than two proteins. Increase "
        "PROTEINS_PER_EC in notebook 01 so that class can appear in both train and test."
    )

# Prefer a stratified grouped 20% holdout when every class has enough support.
if class_counts.min() >= 5:
    splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    train_idx, test_idx = next(splitter.split(np.zeros(len(y)), y, groups))
    split_strategy = "StratifiedGroupKFold first fold"
else:
    warnings.warn(
        "At least one EC class has fewer than five samples; choosing a grouped "
        "holdout that retains every class in training. Some rare classes may not "
        "appear in test.",
        stacklevel=2,
    )
    splitter = GroupShuffleSplit(
        n_splits=100,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
    )
    candidates = list(splitter.split(np.zeros(len(y)), y, groups))
    all_labels = set(y)
    valid = [pair for pair in candidates if set(y[pair[0]]) == all_labels]
    if not valid:
        raise ValueError("Could not create a train split containing every EC class")
    train_idx, test_idx = min(
        valid,
        key=lambda pair: (
            -len(set(y[pair[1]])),
            abs(len(pair[1]) / len(y) - TEST_SIZE),
        ),
    )
    split_strategy = "best class-covered GroupShuffleSplit"

X_train = [graphs[i] for i in train_idx]
X_test = [graphs[i] for i in test_idx]
y_train, y_test = y[train_idx], y[test_idx]
print(f"strategy={split_strategy}; train={len(train_idx)} test={len(test_idx)}")
print("train labels:\n", pd.Series(y_train).value_counts().sort_index())
print("test labels:\n", pd.Series(y_test).value_counts().sort_index())
print("group overlap:", len(set(groups[train_idx]) & set(groups[test_idx])))

In [ ]:
steps = [
    ("protein_labels", ProteinLabelGraphicalizer(alphabet=NODE_LABEL_ALPHABET)),
    (
        "simplex_pair",
        SimplexPairGraphVectorizer(
            max_simplex_dimension=MAX_SIMPLEX_DIMENSION,
            min_distance=MIN_PAIR_DISTANCE,
            max_distance=MAX_PAIR_DISTANCE,
            nbits=NBITS,
            dense=False,
            include_self=True,
            n_jobs=-1,
        ),
    ),
]
if BASE_ESTIMATOR_KIND == "linear_sgd":
    steps.extend([
        ("normalize", Normalizer(norm="l2")),
        ("classifier", SGDClassifier(
            loss="hinge",
            alpha=1e-4,
            max_iter=5000,
            tol=1e-4,
            class_weight="balanced",
            average=True,
            random_state=RANDOM_STATE,
        )),
    ])
elif BASE_ESTIMATOR_KIND == "random_forest":
    steps.append(("classifier", RandomForestClassifier(
        n_estimators=500,
        max_features="sqrt",
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )))
else:
    raise ValueError(f"Unknown BASE_ESTIMATOR_KIND={BASE_ESTIMATOR_KIND!r}")

model = Pipeline(steps)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
labels = sorted(set(y))
report_df = pd.DataFrame(classification_report(
    y_test, y_pred, labels=labels, output_dict=True, zero_division=0
)).T
print(report_df)
print("accuracy:", accuracy_score(y_test, y_pred))
print("balanced accuracy:", balanced_accuracy_score(y_test, y_pred))
confusion_df = pd.DataFrame(
    confusion_matrix(y_test, y_pred, labels=labels),
    index=[f"true_{label}" for label in labels],
    columns=[f"pred_{label}" for label in labels],
)
display(confusion_df)

In [ ]:
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
run_dir = RESULTS_DIR / f"ec_level_{EC_LEVEL}_{run_id}"
run_dir.mkdir(parents=True, exist_ok=True)

predictions = dataset.iloc[test_idx][
    ["sample_id", "pdb_id", "label_asym_id", "uniprot_accession", "ec_number", "protein_name", "graph_path"]
].copy()
predictions["y_true"] = y_test
predictions["y_pred"] = y_pred
predictions["correct"] = predictions["y_true"] == predictions["y_pred"]
predictions.to_csv(run_dir / "predictions.tsv", sep="\t", index=False)
report_df.to_csv(run_dir / "classification_report.tsv", sep="\t")
confusion_df.to_csv(run_dir / "confusion_matrix.tsv", sep="\t")

graphicalizer = model.named_steps["protein_labels"]
metadata = {
    "created_at_utc": run_id,
    "ec_level": EC_LEVEL,
    "target": "ec_number",
    "split_strategy": split_strategy,
    "test_size_requested": TEST_SIZE,
    "random_state": RANDOM_STATE,
    "node_label_alphabet": NODE_LABEL_ALPHABET,
    "max_simplex_dimension": MAX_SIMPLEX_DIMENSION,
    "min_pair_distance": MIN_PAIR_DISTANCE,
    "max_pair_distance": MAX_PAIR_DISTANCE,
    "nbits": NBITS,
    "base_estimator_kind": BASE_ESTIMATOR_KIND,
    "train_samples": int(len(train_idx)),
    "test_samples": int(len(test_idx)),
    "train_test_group_overlap": sorted(set(groups[train_idx]) & set(groups[test_idx])),
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "balanced_accuracy": float(balanced_accuracy_score(y_test, y_pred)),
}
(run_dir / "run_metadata.json").write_text(json.dumps(metadata, indent=2, sort_keys=True) + "\n", encoding="utf-8")
joblib.dump(model, run_dir / "model.joblib")
predictions.head()

The selected UniProt accession is also the split group. The default selector takes one chain per UniProt protein, but retaining grouped splitting protects against future manifests containing repeated structures. A rare-class fallback can produce an incomplete test class mix; inspect the printed support counts before interpreting metrics.